# Ders 1A — Feature Store Oluşturma

Ham veri setleri okunur, moleküler feature dosyaları üretilir ve sınıf dağılımları görselleştirilir.

## 1. Paket kontrolü

Bu hücre, gerekli Python paketlerinin kurulu olup olmadığını kontrol eder. Paket zaten kuruluysa tekrar kurulmaz. Eksikse Colab ortamına otomatik kurulur.

In [ ]:
import sys  # Mevcut Python ortamında pip çalıştırmak için kullanılır.
import subprocess  # Paket kurulum komutlarını yürütmek için kullanılır.
import importlib.util  # Paket kurulu mu kontrol etmek için kullanılır.

def install_if_missing(import_name, pip_name=None):
    """Eksik paket varsa kurar; paket zaten varsa hiçbir şey yapmaz."""
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        print(f"[INSTALL] {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

required_packages = [
    ("pandas", "pandas"),          # CSV okuma ve tablo işlemleri.
    ("numpy", "numpy"),            # Sayısal matris işlemleri.
    ("matplotlib", "matplotlib"),  # Grafik çizimleri.
    ("sklearn", "scikit-learn"),   # Makine öğrenmesi modelleri.
    ("joblib", "joblib"),          # Model kaydetme.
    ("scipy", "scipy"),            # Random search dağılımları.
]

for import_name, pip_name in required_packages:
    install_if_missing(import_name, pip_name)

if importlib.util.find_spec("rdkit") is None:
    try:
        install_if_missing("rdkit", "rdkit")
    except Exception:
        install_if_missing("rdkit", "rdkit-pypi")

print("Paket kontrolü tamamlandı.")

## 2. Ortak yardımcı fonksiyonların yüklenmesi

Bu hücre `molfest_utils.py` dosyasını GitHub raw linkinden indirir ve ortak fonksiyonları çağırır. Böylece her notebookta uzun yardımcı kod bloğu görünmez; ana akış daha okunabilir kalır.

In [ ]:
from pathlib import Path  # Yardımcı dosyanın varlığını kontrol etmek için kullanılır.
import urllib.request  # GitHub raw dosyasını indirmek için kullanılır.
import importlib  # İndirilen modülü yeniden yüklemek için kullanılır.

UTILS_URL = "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_utils.py"  # Ortak fonksiyonların bulunduğu GitHub raw dosyası.

# Notebook içinde uzun yardımcı kod göstermek yerine molfest_utils.py dosyası kısa bir hücreyle indirilir.
urllib.request.urlretrieve(UTILS_URL, "molfest_utils.py")

import molfest_utils  # Ortak fonksiyonlar import edilir.
importlib.reload(molfest_utils)  # Colab eski cache kullanmasın diye modül yenilenir.
from molfest_utils import *  # Notebookta kullanılacak yardımcı fonksiyonlar çalışma alanına alınır.

ensure_dirs()  # Çıktı klasörleri hazırlanır.

print("Ortak yardımcı fonksiyonlar yüklendi.")
print("Feature CSV kaynağı:", FEATURE_BASE_URL)

## 3. Feature store oluşturma

Bu adım ham CSV dosyalarını okur ve feature CSV dosyalarını üretir. Sonraki notebooklar tekrar feature üretmez; hazır feature dosyalarını kullanır.

In [ ]:
feature_index = run_step_01_feature_store()  # Feature CSV dosyalarını üretir, özet tabloyu ve sınıf dağılımı grafiklerini gösterir.

## 4. Üretilen dosyaların kontrolü

Aşağıdaki hücre, GitHub'a yüklenecek `molfest_feature_store/` klasöründeki dosyaları gösterir.

In [ ]:
print("GitHub'a yüklenecek klasör:", FEATURE_STORE.resolve())

for path in sorted(FEATURE_STORE.glob("*")):
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"- {path.name} | {size_mb:.2f} MB")